# NB04 — Conditioned Pairs Portfolio & Ablation

**Role.** The portfolio-evaluation layer. It reads the frozen NB02 statistical baseline and the
NB03 conditioning decisions, and asks — at the portfolio level, net of costs — what the risk overlay
actually does. The plan is a three-variant ablation, **OU / OU+earnings / OU+earnings+sentiment**,
each pushed through the same risk-normalization machinery, then stratified by news coverage and swept
across the earnings horizon `K`.

**Relationship to NB02b.** This promotes the appendix's risk-normalization toolkit — inverse-vol and
causal expanding-vol weighting, the vol-target overlay, breadth (effective-N) and contribution
diagnostics — and runs it on the **conditioned** book rather than the raw baseline. NB02b stays as the
unconditioned bench; NB04 is the deployable read.

---

## Stage 1 — Foundation & reconciliation diagnostic

A portfolio read needs a **daily** PnL series (vol, Sharpe, drawdown all live on it), so conditioning
must be expressed on the daily basis. But NB02 carries two different PnL accountings: the daily
`pnl` series in `pair_timeseries`, and the realized `trade_pnl` in `trades_table`. Before building any
ablation on the daily series, this stage measures their relationship:

- **(A) Orphan check** — how much daily PnL sits *outside* every trade window. The overlay can only
  touch bars it owns (bars inside a trade), so if a large share of daily PnL is un-owned, daily-basis
  conditioning would reach only a fraction of the book — a problem to understand *before* any ablation
  number exists. If the orphan slice is negligible, the daily basis is fully conditionable and we
  proceed.
- **(B) Per-trade tie-out** — whether summing daily PnL over a trade's `[entry, exit]` window recovers
  that trade's `trade_pnl`. If it does, conditioning the daily windows also exactly removes realized
  PnL (a clean dual reading). If it does not, the portfolio simply lives on the daily-MTM basis — the
  correct basis for risk metrics regardless — and the trade-level totals are documented as a separate
  cut, not forced to match.

### 1.0 Load the frozen baseline and the conditioning decisions

NB02 artifacts (`pair_timeseries`, `trades_table`, `pairs_metadata`) and NB03 outputs (`trades_conditioned`, `semantic_evidence`) are loaded read-only. The daily per-pair PnL pivots into a `date × pair` matrix on NB02's full OOS index, with gaps already zero-filled. The schema of `trades_conditioned` is printed so the columns are confirmed at a glance before anything is built on them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def find_project_root(markers=("pyproject.toml", ".git")):
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if any((cand / m).exists() for m in markers):
            return cand
    return here

ROOT     = find_project_root()
NB2_DIR  = ROOT / "artifacts" / "nb2_outputs"
NB3_DIR  = ROOT / "artifacts" / "nb3_outputs"
ANNUAL_D = 252

# --- frozen NB02 baseline (read-only) ---
pair_ts = pd.read_parquet(NB2_DIR / "pair_timeseries.parquet")
trades  = pd.read_csv(NB2_DIR / "trades_table.csv", parse_dates=["entry_date", "exit_date"])
meta    = pd.read_csv(NB2_DIR / "pairs_metadata.csv")

# --- NB03 conditioning decisions (read-only) ---
trades_cond = pd.read_csv(NB3_DIR / "trades_conditioned.csv", parse_dates=["entry_date", "exit_date"])
evidence    = pd.read_parquet(NB3_DIR / "semantic_evidence.parquet")
earnings_raw = pd.read_parquet(NB3_DIR / "earnings_raw.parquet")   # raw calendar, for the Stage-3 K-sweep

# Daily per-pair PnL matrix (date x pair). NB02 exports pnl = gross - tc on the full OOS index.
pnl_wide = (pair_ts.pivot(index="date", columns="pair", values="pnl").sort_index().fillna(0.0))
pnl_wide.index = pd.to_datetime(pnl_wide.index).normalize()

# NB03 policy multipliers (the action -> position-scale map)
MULT = {"ALLOW": 1.0, "DOWNSIZE": 0.5, "BLOCK": 0.0}

print(f"[OK] pnl_wide          : {pnl_wide.shape}  ({pnl_wide.shape[1]} pairs x {len(pnl_wide)} days)")
print(f"[OK] trades (NB2)      : {trades.shape}")
print(f"[OK] trades_cond (NB3) : {trades_cond.shape}")
print(f"     columns           : {list(trades_cond.columns)}")
print(f"[OK] action mix        : {trades_cond['action'].value_counts().to_dict()}")
_missing = sorted(set(trades_cond['pair']) - set(pnl_wide.columns))
print(f"[CHECK] trade pairs absent from pnl_wide: {_missing if _missing else 'none'}")
print(f"[CHECK] OOS span        : {pnl_wide.index.min().date()} -> {pnl_wide.index.max().date()}")

### 1.1 Bar-ownership map + reconciliation

For each pair-date the active trade (if any) is identified, giving a `date × pair` boolean **in_trade** mask and a parallel **multiplier** matrix (the action's position scale, `1.0` where no trade is open). One position per pair at a time, so a bar has at most one owner. The two checks above are then computed directly off these.

In [ ]:
# ---- Bar-ownership map: which trade (if any) is active on each pair-date ----
in_trade  = pd.DataFrame(False, index=pnl_wide.index, columns=pnl_wide.columns)
mult_wide = pd.DataFrame(1.0,   index=pnl_wide.index, columns=pnl_wide.columns)   # 1.0 = no trade open

for r in trades_cond.itertuples(index=False):
    if r.pair not in in_trade.columns:
        continue
    m = (in_trade.index >= r.entry_date) & (in_trade.index <= r.exit_date)
    in_trade.loc[m, r.pair] = True
    mult_wide.loc[m, r.pair] = MULT.get(r.action, 1.0)

# ---- (A) inside vs outside trade windows --------------------------------------------------
total   = float(pnl_wide.values.sum())
inside  = float(np.nansum(pnl_wide.where(in_trade).values))
outside = total - inside
gross   = float(np.abs(pnl_wide.values).sum())
held_share = float(in_trade.values.mean())

print("=== (A) Daily PnL: inside vs outside trade windows ===")
print(f"  bars inside a trade window : {held_share:.1%} of all pair-days")
print(f"  Sigma all daily pnl        : {total:14,.2f}")
print(f"  Sigma inside trade windows : {inside:14,.2f}")
print(f"  Sigma outside (orphan)     : {outside:14,.2f}"
      f"   ({(outside/total if total else 0):+.1%} of total, "
      f"{abs(outside)/gross if gross else 0:.2%} of gross activity)")

# ---- (B) per-trade tie-out: daily window-sum vs realized trade_pnl -------------------------
rows = []
for r in trades_cond.itertuples(index=False):
    if r.pair not in pnl_wide.columns:
        continue
    m = (pnl_wide.index >= r.entry_date) & (pnl_wide.index <= r.exit_date)
    rows.append((r.pair, float(r.trade_pnl), float(pnl_wide.loc[m, r.pair].sum())))
recon = pd.DataFrame(rows, columns=["pair", "trade_pnl", "daily_window_sum"])
recon["diff"] = recon["daily_window_sum"] - recon["trade_pnl"]

print("\n=== (B) Per-trade tie-out: Sigma(daily pnl over [entry,exit]) vs trade_pnl ===")
print(f"  Sigma trade_pnl            : {recon['trade_pnl'].sum():14,.2f}")
print(f"  Sigma daily_window_sum     : {recon['daily_window_sum'].sum():14,.2f}")
print(f"  trades tying out (<1e-6)   : {(recon['diff'].abs() < 1e-6).sum()} / {len(recon)}")
print(f"  median |per-trade diff|    : {recon['diff'].abs().median():14,.4f}")
print(f"  max    |per-trade diff|    : {recon['diff'].abs().max():14,.4f}")

### 1.2 The diagnostic, at a glance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# (A) inside vs outside
ax[0].bar(["inside trades", "orphan (outside)"], [inside, outside],
          color=["#3b6ea5", "#c0504d"])
ax[0].axhline(0, color="k", lw=1)
ax[0].set_title("(A) Daily PnL ownership")
ax[0].set_ylabel("Sigma daily pnl (spread-$)")
for i, v in enumerate([inside, outside]):
    ax[0].text(i, v, f"{v:,.0f}", ha="center", va="bottom" if v >= 0 else "top")

# (B) per-trade tie-out distribution (spike at 0 == clean tie-out)
ax[1].hist(recon["diff"], bins=40, color="#8064a2")
ax[1].axvline(0, color="k", ls="--", lw=1)
ax[1].set_title("(B) Per-trade diff: daily window-sum - trade_pnl")
ax[1].set_xlabel("difference (spread-$)"); ax[1].set_ylabel("trades")

plt.tight_layout(); plt.show()

### How to read this — and what Stage 2 does next

**(A) is the gate.** If the orphan slice is negligible (a few percent of gross activity at most),
daily-basis conditioning reaches essentially the entire book: scaling a trade's daily bars by its
action multiplier — `0` for BLOCK, `0.5` for DOWNSIZE, `1` for ALLOW — faithfully removes or shrinks
that position's contribution, and the ablation rests on solid ground. If instead a large share of
daily PnL sits *outside* all trade windows, the daily series carries exposure the trade decisions
cannot reach, and that gap must be understood before any ablation number is trusted — that is the
finding to surface, not paper over.

**(B) characterizes the two accountings.** A clean spike at zero means each trade's realized PnL is
exactly its daily window-sum, so the daily and trade-level views agree and either can be quoted. A
spread away from zero means they are simply different cuts of the same positions; the portfolio then
lives on the daily-MTM basis (the right basis for vol and drawdown anyway), and we will report
trade-level totals separately rather than force a reconciliation that NB02's design never promised.

Either way, **`mult_wide` is the object Stage 2 builds on**: the conditioned daily PnL for a variant is
just `pnl_wide` times the variant's multiplier matrix. OU is all-ones; OU+earnings zeroes the
earnings-blocked windows; OU+earnings+sentiment additionally halves the sentiment-downsized ones. Each
variant then flows through the promoted NB02b machinery — inverse-vol and causal weighting, vol-target,
breadth, contribution — for an apples-to-apples comparison on one shared daily basis.

---

## Stage 2 — The three-variant ablation

With the daily basis verified, the real question is what the overlay actually *does* to the book. The
answer is built as a ladder, where each rung adds exactly one decision — so any change is attributable
to that decision and nothing else:

- **OU** — the unconditioned baseline: every position held exactly as NB02 traded it.
- **OU + earnings** — the earnings gate only: positions that would be held through a scheduled report
  are flattened.
- **OU + earnings + sentiment** — the full policy: earnings-blocked positions flattened, adverse-
  sentiment positions halved.

All three sit on the *same* daily PnL and the *same* fixed pair weights, so the only thing moving from
one column to the next is the conditioning itself.

One expectation is worth setting before the numbers arrive. Stage 1 showed the earnings-blocked half of
the book is roughly **break-even** in realized terms (−18.8 out of 310). So if the earnings gate earns
its keep, it will not announce itself in a higher average return — it will show up in the **shape** of
the distribution: fewer brutal days, a shallower worst drawdown, a less negative skew. A mean-reversion
position held *through* an earnings jump is exactly the kind of trade that occasionally goes very wrong;
removing those should trim the left tail far more than it touches the middle. So this stage reads the
ladder through **drawdown, skew, and the left tail** — not the headline Sharpe, which can sit still
even when the risk profile genuinely improves.

### 2.1 Bar ownership and the three conditioning matrices

Each pair-date is assigned to exactly one trade. On the rare overlap — a flip day where one trade's exit meets the next trade's entry — the later-entering trade owns the bar, since that is the position actually carried forward. From those owners, three multiplier matrices follow: all-ones for OU; earnings-blocked windows zeroed for OU+earnings; and the full policy (block → 0, downsize → 0.5) for the last.

In [ ]:
def variant_multipliers(tc, index, columns):
    """Three date x pair multiplier matrices, one per ablation rung.
    Overlap rule: later-entering trade owns the bar (sorted ascending, last write wins)."""
    ou   = pd.DataFrame(1.0, index=index, columns=columns)   # baseline: no conditioning
    earn = pd.DataFrame(1.0, index=index, columns=columns)    # earnings gate only
    full = pd.DataFrame(1.0, index=index, columns=columns)    # earnings + sentiment
    POL = {"ALLOW": 1.0, "DOWNSIZE": 0.5, "BLOCK": 0.0}
    for r in tc.sort_values("entry_date").itertuples(index=False):
        if r.pair not in columns:
            continue
        m = (index >= r.entry_date) & (index <= r.exit_date)
        earn.loc[m, r.pair] = 0.0 if r.action == "BLOCK" else 1.0   # BLOCK == earnings, by construction
        full.loc[m, r.pair] = POL.get(r.action, 1.0)
    return ou, earn, full

mult_ou, mult_earn, mult_full = variant_multipliers(trades_cond, pnl_wide.index, pnl_wide.columns)
pnl_OU   = pnl_wide * mult_ou
pnl_EARN = pnl_wide * mult_earn
pnl_FULL = pnl_wide * mult_full

# how much of the held book each layer touches (share of in-trade bars scaled away from 1.0)
held = in_trade.values
print(f"[layers] in-trade bars flattened by earnings gate : "
      f"{(mult_earn.values[held] == 0).mean():.1%}")
print(f"[layers] in-trade bars halved by sentiment gate   : "
      f"{(mult_full.values[held] == 0.5).mean():.1%}")
print(f"[layers] in-trade bars left untouched (ALLOW)      : "
      f"{(mult_full.values[held] == 1.0).mean():.1%}")

### 2.2 Fixed risk weights, and the portfolio read

Raw spread-dollar PnL cannot be averaged across pairs directly: the hedge ratio β spans orders of magnitude, so a plain mean is dominated by a few large-scale names rather than reflecting the book. Each pair is therefore weighted by its **inverse volatility**, so every pair contributes comparable risk. Those weights are computed **once on the unconditioned book and held fixed** across all three variants — the only thing that changes column to column is the conditioning, never a re-optimization of the weights. The summary adds the three columns that matter most for this question: **skew**, the **left-tail** average (mean of the worst 5% of days), and the active-day **hit rate**.

In [ ]:
def port_stats(pnl, label, annual=ANNUAL_D):
    pnl = pnl.fillna(0.0)
    mu, vol = pnl.mean() * annual, pnl.std(ddof=0) * np.sqrt(annual)
    eq = pnl.cumsum()
    nz = pnl[pnl != 0.0]
    k  = max(1, int(0.05 * len(nz)))
    return {
        "Variant":       label,
        "AnnPnL(norm)":  mu,
        "AnnVol":        vol,
        "Sharpe":        (mu / vol) if vol > 0 else np.nan,
        "MaxDD":         float((eq - eq.cummax()).min()),
        "Skew":          float(pnl.skew()),
        "LeftTail5%":    float(nz.nsmallest(k).mean()) if len(nz) else np.nan,  # mean of worst 5% days
        "HitRate%":      100.0 * float((nz > 0).mean()) if len(nz) else np.nan,
    }

# fixed inverse-vol weights from the UNCONDITIONED book
pair_vol = pnl_wide.std(ddof=0).replace(0.0, np.nan)
w = (1.0 / pair_vol); w = (w / w.sum()).fillna(0.0)
portfolio = lambda pnl_mat: (pnl_mat * w).sum(axis=1)

port_OU, port_EARN, port_FULL = portfolio(pnl_OU), portfolio(pnl_EARN), portfolio(pnl_FULL)

summary = pd.DataFrame([
    port_stats(port_OU,   "OU (baseline)"),
    port_stats(port_EARN, "OU + earnings"),
    port_stats(port_FULL, "OU + earnings + sentiment"),
]).set_index("Variant")

print("Three-variant ablation — inverse-vol weighted, weights fixed on the baseline book:")
print(summary.round(3).to_string())

### 2.3 Equity and drawdown

The cumulative path and the underwater (drawdown) curve, all three variants on one comparable risk-normalized scale. The drawdown panel is the one to watch — it is where flattening the earnings-spanning trades should show, if it shows anywhere.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for s, lab, c in [(port_OU, "OU (baseline)", "#7f7f7f"),
                  (port_EARN, "OU + earnings", "#3b6ea5"),
                  (port_FULL, "OU + earnings + sentiment", "#c0504d")]:
    eq = s.cumsum()
    ax[0].plot(eq.index, eq.values, label=lab, lw=1.8)
    dd = eq - eq.cummax()
    ax[1].plot(dd.index, dd.values, label=lab, lw=1.4)
ax[0].set_title("Cumulative PnL (risk-normalized)"); ax[0].set_ylabel("cumulative norm PnL"); ax[0].legend(fontsize=8)
ax[1].set_title("Underwater / drawdown"); ax[1].set_ylabel("drawdown (norm PnL)"); ax[1].axhline(0, color="k", lw=1)
plt.tight_layout(); plt.show()

### Reading the ladder

The columns to compare are the tail ones. If the earnings gate is doing real work, **OU + earnings**
should show a **shallower MaxDD and a less-negative skew / left-tail** than the baseline, *even if its
Sharpe barely moves* — that is the signature of a gate that removes jump risk rather than chasing
return. The sentiment column then layers its small, coverage-bound adjustment on top of that.

If instead the tail columns sit unchanged, the honest reading is that the gate removed a break-even
slice of trades without improving the risk profile — and that, too, is a finding, one the K-sweep in
Stage 3 will stress-test by asking whether a different earnings horizon changes the verdict. Either
way the answer lives in the tail and the drawdown, exactly where Stage 1 pointed — not in a headline
average that the break-even result already told us would stay quiet.

---

## Stage 3 — Where it works, and how robustly

Stage 2 left two open questions, and they decide how much to trust the result.

The first is about **sentiment**. Overall it was irrelevant — but "overall" averages the dense names in
with the silent ones. The honest test is to split the book by coverage and look only at the slice where
the gate can actually see: on the covered names, does adverse-sentiment downsizing help, hurt, or still
do nothing? The distinction matters. If sentiment works *where it can see* and merely runs out of data
elsewhere, the limit is coverage — a data-breadth problem. If it fails even on the covered names, the
signal itself is weak, and more data would not rescue it.

The second is about **earnings**. The gate was fixed at K=33 — the median hold — on principle, before
any result was seen, so it is not tuned to the outcome. But one horizon is still one point. Sweeping K
across {7, 14, 21, 33, 56} shows whether the risk reduction is a stable feature of the strategy or a
knife-edge at a single value.

### 3.1 The ablation, split by coverage

Each trade is tagged **COVERED** if either leg cleared the evidence floor at entry, **THIN** otherwise. The two strata are the covered and thin *slices of the same fixed-weight book* — not re-optimized sub-funds — so the conditioning stays the only moving part. By construction the sentiment gate can act only on covered trades, so on the thin slice the last two columns must coincide; the covered slice is where any sentiment effect has to show.

In [ ]:
# bar -> owning trade_id (later-entering trade wins on overlap), sentinel -1 for no trade
owner = pd.DataFrame(np.nan, index=pnl_wide.index, columns=pnl_wide.columns)
for r in trades_cond.sort_values("entry_date").itertuples(index=False):
    if r.pair not in owner.columns:
        continue
    m = (owner.index >= r.entry_date) & (owner.index <= r.exit_date)
    owner.loc[m, r.pair] = r.trade_id
owner = owner.fillna(-1).astype(int)

# coverage stratum per trade: COVERED if any leg cleared the evidence floor
cov = evidence.groupby("trade_id")["evidence_status"].apply(lambda s: (s == "COVERED").any())
covered_ids = [int(t) for t, c in cov.items() if c]
thin_ids    = [int(t) for t, c in cov.items() if not c]
print(f"[strata] COVERED trades: {len(covered_ids)}   THIN trades: {len(thin_ids)}")

def stratum_table(ids):
    mask = owner.isin(list(ids))
    out = []
    for name, mult in [("OU (baseline)", mult_ou),
                       ("OU + earnings", mult_earn),
                       ("OU + earnings + sentiment", mult_full)]:
        port = (((pnl_wide * mult).where(mask, 0.0)) * w).sum(axis=1)
        out.append(port_stats(port, name))
    return out

rows = []
for strat, ids in [("COVERED", covered_ids), ("THIN", thin_ids)]:
    for d in stratum_table(ids):
        d["Stratum"] = strat
        rows.append(d)
strat_summary = pd.DataFrame(rows).set_index(["Stratum", "Variant"])
print("\nCoverage-stratified ablation (slices of the same fixed-weight book):")
print(strat_summary[["Sharpe", "AnnVol", "MaxDD", "Skew", "LeftTail5%", "HitRate%"]].round(3).to_string())

### Reading the split

Two things to check. On the **THIN** slice the earnings and earnings+sentiment rows should be
identical — sentiment has nothing to act on, by construction; if they differ, something is mislabelled.
On the **COVERED** slice, compare those same two rows: a *lower* Sharpe / heavier tail under the
sentiment column means the gate hurts even where it can see (a weak signal, not just a coverage gap); an
*improvement* means sentiment does carry information on the dense names and the only thing stopping it
elsewhere is the absence of news — a data-breadth limit, not a dead end. The covered slice is small, so
this is a directional read, not a precise estimate — which is itself part of the finding.

### 3.3 Robustness to the earnings horizon K

The earnings gate is re-derived from scratch at each horizon — a trade is blocked if either leg has a scheduled report within K days of entry — straight from the raw earnings calendar, so NB03 stays frozen. For each K the book is rebuilt and its risk profile recomputed. A built-in cross-check: the recomputed block count at K=33 should land on the figure NB03 reported.

In [ ]:
# raw earnings dates per bare ticker (defensive on the persisted schema)
er = earnings_raw.copy()
if "ticker" not in er.columns and "code" in er.columns:
    er["ticker"] = er["code"].astype(str).str.replace(r"\.US$", "", regex=True)
er["report_date"] = pd.to_datetime(er["report_date"], errors="coerce")
earn_dates = {tk: np.sort(g["report_date"].dropna().values)
              for tk, g in er.dropna(subset=["report_date"]).groupby("ticker")}
leg_map = meta.set_index("pair")[["Ticker1", "Ticker2"]]

def blocked_ids_at_K(K):
    ids = set()
    for r in trades_cond.itertuples(index=False):
        if r.pair not in leg_map.index:
            continue
        ent = pd.Timestamp(r.entry_date)
        lo, hi = np.datetime64(ent), np.datetime64(ent + pd.Timedelta(days=int(K)))
        for tk in (leg_map.loc[r.pair, "Ticker1"], leg_map.loc[r.pair, "Ticker2"]):
            d = earn_dates.get(tk)
            if d is not None and d.size and np.any((d >= lo) & (d <= hi)):
                ids.add(int(r.trade_id)); break
    return ids

K_GRID = [7, 14, 21, 33, 56]
sweep = []
for K in K_GRID:
    bids  = blocked_ids_at_K(K)
    bmask = owner.isin(list(bids))
    port  = ((pnl_wide.where(~bmask, 0.0)) * w).sum(axis=1)
    d = port_stats(port, f"K={K}")
    d["K"] = K
    d["n_blocked"] = len(bids)
    sweep.append(d)
sweepdf = pd.DataFrame(sweep).set_index("K")
print("Earnings-horizon sweep (OU + earnings only, fixed weights):")
print(sweepdf[["n_blocked", "Sharpe", "AnnVol", "MaxDD", "Skew", "LeftTail5%"]].round(3).to_string())
print(f"\n[check] recomputed blocks at K=33: {sweepdf.loc[33, 'n_blocked']}  (NB03 reported 139)")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].plot(sweepdf.index, sweepdf["Sharpe"], marker="o", color="#3b6ea5", lw=2)
ax[0].axvline(33, color="gray", ls="--", lw=1); ax[0].text(33, ax[0].get_ylim()[0], " K=33", color="gray", va="bottom")
ax[0].set_title("Sharpe vs earnings horizon K"); ax[0].set_xlabel("K (calendar days)"); ax[0].set_ylabel("Sharpe")
ax[1].plot(sweepdf.index, sweepdf["LeftTail5%"], marker="o", color="#c0504d", lw=2, label="LeftTail5%")
ax[1].plot(sweepdf.index, sweepdf["MaxDD"], marker="s", color="#8064a2", lw=1.6, label="MaxDD")
ax[1].axvline(33, color="gray", ls="--", lw=1)
ax[1].set_title("Left tail & drawdown vs K"); ax[1].set_xlabel("K (calendar days)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Reading the sweep

The cross-check lands exactly — the gate recomputed from the raw calendar blocks **139** trades at
K=33, matching NB03 — so the curve can be trusted. And its shape is clear. **Every horizon from a week
up to the median hold lifts Sharpe well above the unconditioned 0.72 baseline** — from ~1.14 at K=7 to
a peak near **1.29 at K=21** — with the left tail and drawdown improving alongside. The risk reduction
is therefore a *robust feature* of conditioning on earnings across a wide band, not a knife-edge at one
value. Only at **K=56**, where the gate stands aside on two-thirds of the book (186 of 280 trades), does
over-restriction erase the benefit and pull Sharpe below baseline — the strategy blocked into the
ground.

One honest note on K. The sweep makes plain that K=33 is *not* the in-sample optimum; a shorter window
(~21 days) scored higher here. K is deliberately left at 33 — the median holding period, fixed on
principle before any of these numbers were seen — rather than slid to the peak, because choosing K to
maximize the Sharpe it produces is precisely the outcome-tuning this study exists to avoid. The value of
the sweep is the *shape* — broad, robust benefit that comfortably contains the principled choice — not a
licence to pick the winner.

## Conclusions

Read end to end, the conditioned book tells a two-part story, and both parts are honest.

**The earnings gate is the substance, and it is robust.** Flattening the trades that would be held
through a scheduled report removes a slice of the book that was close to break-even in return but
carried real jump risk — so the cost in return is small while the cut in volatility, drawdown, and
left-tail severity is large, lifting Sharpe from 0.72 to roughly 1.0 at the chosen horizon. Because
Sharpe is leverage-invariant, that is a genuine improvement in return-per-unit-risk, not an artifact of
trading less. The horizon sweep confirms it is no knife-edge: every K from a week to the median hold
beats the baseline comfortably, the benefit peaking near three weeks and collapsing only once the gate
over-blocks two-thirds of the book. K was fixed at the median hold on principle and deliberately left
there, not slid to the in-sample peak.

**The sentiment gate is bounded twice over, and the boundary is the result.** First by coverage: for
most of this Latin-American ADR universe there is too little news, as-of entry, to form a view, so the
gate abstains and barely moves the portfolio. Second — and more telling — by signal: on the covered
slice, where the gate *can* see, downsizing on adverse tone slightly *reduced* risk-adjusted return
rather than improving it, across the handful of trades it touched. So the limit is not only that the
news is thin; it is that the tone signal, as available here, did not single out the trades worth
shrinking. That distinction shapes what would move the picture — broader, local-language or quant-grade
sentiment data would raise coverage and finally make the signal *testable* at scale, but the early
evidence leans toward a weak signal, not merely a sparse one. Drawing that boundary precisely, rather
than claiming an edge the data cannot support, is the contribution.

What belongs to a later stage: a true capital-scaled allocation (equal-capital percent returns, which
need per-pair notional reconstructed from prices), an explicit stop-loss, and broader sentiment sourcing
to turn the sentiment question from untestable into tested.